In [9]:
# 06_brinson_sector_spy.py
from pathlib import Path
import pandas as pd
import numpy as np

# paths
RET_ASSET = Path('data/processed/returns_linear.parquet')      # 13 tickers + SPY
W_WIDE    = Path('data/portfolio_weights/custom_monthly_wide.parquet')  # 13 tickers
META      = Path('data/raw/meta_tickers.csv')                  # ticker,sector
SEC_RET   = Path('data/raw/sector_etf_returns.parquet')        # sector ETF returns (R_s^B)
WB_DAILY  = Path('data/processed/spy_sector_weights_daily.parquet') # w_s^B daily
OUT_ATTR  = Path('data/processed/attribution_results_sector_spy.parquet')      # overwrite with sector-based
OUT_CHECK = Path('data/processed/active_returns_from_weights.parquet')        # optional

def to_sector_portfolio(ret_assets: pd.DataFrame, w_wide: pd.DataFrame, meta: pd.DataFrame):
    sec_map = meta.set_index('ticker')['sector'].to_dict()
    sectors = sorted(set(sec_map.get(t,'Unknown') for t in ret_assets.columns))
    # sector weight (portfolio top-level)
    w_sec = pd.DataFrame(index=w_wide.index, columns=sectors, data=0.0)
    r_sec = pd.DataFrame(index=ret_assets.index, columns=sectors, data=0.0)
    for s in sectors:
        tick = [t for t in ret_assets.columns if sec_map.get(t) == s]
        if not tick: continue
        w_sub = w_wide[tick]
        w_sec[s] = w_sub.sum(axis=1)                      # top-level sector weight
        intra = w_sub.div(w_sub.sum(axis=1).replace(0,np.nan), axis=0)
        r_sec[s] = (intra * ret_assets[tick]).sum(axis=1) # sector return from assets
    return r_sec, w_sec

def brinson_sector(r_sec_p: pd.DataFrame, w_sec_p: pd.DataFrame,
                   r_sec_b: pd.DataFrame, w_sec_b: pd.DataFrame) -> pd.DataFrame:
    # dùng w(t-1) cho cả port & bench
    wp = w_sec_p.shift(1).reindex(r_sec_p.index).ffill()
    wb = w_sec_b.shift(1).reindex(r_sec_b.index).ffill()
    # R^B
    rB = (wb * r_sec_b).sum(axis=1)
    # components
    alloc = (wp - wb).mul(r_sec_b.sub(rB, axis=0), axis=0).sum(axis=1)
    sel   = wb.mul(r_sec_p - r_sec_b, axis=0).sum(axis=1)
    inter = (wp - wb).mul(r_sec_p - r_sec_b, axis=0).sum(axis=1)
    out = pd.DataFrame({'Allocation': alloc, 'Selection': sel, 'Interaction': inter})
    out['Total'] = out.sum(axis=1)
    return out

def main():
    ret_all = pd.read_parquet(RET_ASSET)              # 13 + SPY
    asset_cols = [c for c in ret_all.columns if c != 'SPY']
    ret_assets = ret_all[asset_cols]

    w_wide = pd.read_parquet(W_WIDE)
    meta   = pd.read_csv(META)
    r_sec_p, w_sec_p = to_sector_portfolio(ret_assets, w_wide, meta)

    # benchmark sector returns & weights (SPY-consistent)
    r_sec_b = pd.read_parquet(SEC_RET).reindex(r_sec_p.index).ffill()
    w_sec_b = pd.read_parquet(WB_DAILY).reindex(r_sec_p.index).ffill()

    # cắt chung cột sector
    common_sec = r_sec_p.columns.intersection(r_sec_b.columns)
    r_sec_p = r_sec_p[common_sec]; w_sec_p = w_sec_p[common_sec]
    r_sec_b = r_sec_b[common_sec]; w_sec_b = w_sec_b[common_sec]

    attr = brinson_sector(r_sec_p, w_sec_p, r_sec_b, w_sec_b)
    OUT_ATTR.parent.mkdir(parents=True, exist_ok=True)
    attr.to_parquet(OUT_ATTR)
    print("Saved:", OUT_ATTR.resolve())

    # kiểm tra: monthly Total vs active (port − SPY)
    ret_p = (w_wide.shift(1).reindex(ret_assets.index).ffill() * ret_assets).sum(axis=1)
    active = (ret_p - ret_all['SPY']).fillna(0)
    chk = pd.DataFrame({
        'Total_M': attr['Total'].resample('ME').sum(),
        'Active_M': active.resample('ME').sum()
    }).dropna()
    chk.to_parquet(OUT_CHECK)
    print("Check file:", OUT_CHECK.resolve())

if __name__ == '__main__':
    main()

Saved: C:\Users\ACER\python\Dissertation\Port Dashboard\data\processed\attribution_results_sector_spy.parquet
Check file: C:\Users\ACER\python\Dissertation\Port Dashboard\data\processed\active_returns_from_weights.parquet


In [5]:
import pandas as pd

w_port = pd.read_parquet("data/portfolio_weights/ew_monthly_wide.parquet")
w_bench = pd.read_parquet("data/processed/spy_sector_weights_daily.parquet")

print("Portfolio weights mean:", w_port.mean().round(3))
print("SPY sector weights mean:", w_bench.mean().round(3))

Portfolio weights mean: AAPL     0.077
MSFT     0.077
NVDA     0.077
GOOGL    0.077
AMZN     0.077
TSLA     0.077
JPM      0.077
JNJ      0.077
UNH      0.077
XOM      0.077
PG       0.077
CAT      0.077
NEE      0.077
dtype: float64
SPY sector weights mean: XLB     0.028
XLC     0.096
XLE     0.047
XLF     0.113
XLI     0.070
XLK     0.264
XLP     0.070
XLRE    0.026
XLU     0.032
XLV     0.132
XLY     0.111
dtype: float64


In [7]:
import pandas as pd, numpy as np

w_wide  = pd.read_parquet("data/portfolio_weights/ew_monthly_wide.parquet")
meta    = pd.read_csv("data/raw/meta_tickers.csv").set_index("ticker")["sector"].to_dict()
wb_spy  = pd.read_parquet("data/processed/spy_sector_weights_daily.parquet")

# gộp danh mục theo sector
sectors = sorted(set(meta.get(t,"Other") for t in w_wide.columns))
w_sec = pd.DataFrame(0.0, index=w_wide.index, columns=sectors)
for s in sectors:
    tick = [t for t in w_wide.columns if meta.get(t)==s]
    if tick:
        w_sec[s] = w_wide[tick].sum(axis=1)

print("Portfolio sector weights (mean):")
print(w_sec.mean().sort_values(ascending=False).round(3))

print("\nSPY sector weights (mean):")
print(wb_spy.mean().sort_values(ascending=False).round(3))

Portfolio sector weights (mean):
XLK    0.231
XLV    0.154
XLY    0.154
XLC    0.077
XLE    0.077
XLF    0.077
XLI    0.077
XLP    0.077
XLU    0.077
dtype: float64

SPY sector weights (mean):
XLK     0.264
XLV     0.132
XLF     0.113
XLY     0.111
XLC     0.096
XLP     0.070
XLI     0.070
XLE     0.047
XLU     0.032
XLB     0.028
XLRE    0.026
dtype: float64
